In [1]:
import os
import csv

from datetime import timedelta, timezone, datetime
from zoneinfo import ZoneInfo
from itertools import groupby

import pytz
import dateutil.parser

from nhk import ProgramGuideV3

In [2]:
def round_time(dt:datetime.date=None, round_to=60)->datetime.date:

	if dt == None:
		dt = datetime.now()

	seconds = (dt.replace(tzinfo=None) - dt.min).seconds
	rounding = (seconds + round_to / 2) // round_to * round_to

	return dt + timedelta(0, rounding - seconds, -dt.microsecond)



def get_publications(client:ProgramGuideV3, area:str, service:str, date:datetime)->list:

	date_radio = client.pg_date_radio(area=area, service='r3', date=date)

	publications = date_radio.get('r3').get('publication')

	return publications


def extract_publications_by_genre(publications:list, genres=None)->list:

	extracted = [pub for pub in publications
		if (genres is None) or
			(set([genre.get('id') for genre in pub.get('identifierGroup').get('genre')])
				.intersection(genres))
	]

	return extracted


def extract_publications_by_date(publications:list, dt_tday:datetime, dt_pivot:datetime)->list:

	extracted = [pub for pub in publications
		if dateutil.parser.parse(pub.get('startDate')) >= dt_tday and
			dateutil.parser.parse(pub.get('startDate')) < dt_pivot
	]

	return extracted


def extract_consecutive_runs(data:list)->list:

    results = []
    for k, g in groupby(enumerate(data), key=lambda x: x[0] - x[1]):
        group_list = [i[1] for i in g]
        results.append(group_list)

    return results


def get_alias_id(broadcast_event:dict)->str:
	alias_id = None
	about = broadcast_event.get('about')
	if about is None:
		return None
	identifier_group = about.get('identifierGroup')
	if identifier_group is None:
		return None
	alias_id = identifier_group.get('aliasId')
	return alias_id


def get_series_id(broadcast_event:dict)->str:
	series_id = None
	identifier_group = broadcast_event.get("identifierGroup")
	if identifier_group is None:
		return None
	series_id = identifier_group.get("radioSeriesId")
	return series_id


def get_series_url(broadcast_event:dict)->str:
	series_url = None
	about = broadcast_event.get('about')
	if about is None:
		return None
	identifier_group = about.get('partOfSeries')
	if identifier_group is None:
		return None
	series_url = identifier_group.get('canonical')
	return series_url


def get_about_url(broadcast_event:dict)->str:
	about_url = None
	about = broadcast_event.get('about')
	if about is None:
		return None
	about_url = about.get('canonical')
	return about_url


def get_start_date(broadcast_event:dict)->str:
	start_date = broadcast_event.get('startDate')
	return start_date


def get_end_date(broadcast_event:dict)->str:
	end_date = broadcast_event.get('endDate')
	return end_date

In [3]:
keytitles = [
    'hikigatari',
    'rewindtime',
    'kogaku',
    'yuran',
    'kayou',
    'opera-f',
    'bescla',
    'ml',
    'viva',
    'endless',
    'nkyolegend',
    'nhkconcert',
    'kaiteki',
    'soul',
    'jazz',
    'hibiki',
    'meiensou',
    '#ijuin100r',
    'kakecla',
    'hoshizora',
    'bravo',
    'r-passio',
    'jazzvoyage',
    'anata',
    'classicmeikyu',
    'bayreuth',
    'maro',
    'discovere',
    'LG96ZW5KZ4',
    '6J686W68QL',
    '89Z6622NZ8',
    '7WJ94ZQ36Y',
    'J9QGZ654YR',
    '885X7WKYRN',
    'J1KJ59Q96X',
    'KZ5Q9LVMWL',
    'MJ885Z5MXR',
    'K1VRXRKXG6',
    'MK7LP4934M'
]

In [4]:
reruns = {
    'r-passio': ['05:00'],
    'kakecla': ['07:25'],
    'endless': ['09:15'],
    'kaiteki': ['16:00'],
    'meiensou': ['16:00'],
    'jazz': ['16:00'],
    'ml': ['10:15'],
    'LG96ZW5KZ4': ['07:25'],
    'yuran': ['04:00'],
    'kayou': ['04:30'],
    'hikigatari': ['09:55'],
    'K1VRXRKXG6': ['09:15', '04:00'],
    'jazzvoyage': ['06:00'],
    'KZ5Q9LVMWL': ['08:10'],
    '6J686W68QL': ['08:10']
}

In [5]:
key_reruns = reruns.keys()
key_reruns

dict_keys(['r-passio', 'kakecla', 'endless', 'kaiteki', 'meiensou', 'jazz', 'ml', 'LG96ZW5KZ4', 'yuran', 'kayou', 'hikigatari', 'K1VRXRKXG6', 'jazzvoyage', 'KZ5Q9LVMWL', '6J686W68QL'])

In [6]:
signs = {
    'LG96ZW5KZ4': 'c-niwa',
    '6J686W68QL': 'gendai',
    '89Z6622NZ8': 'yuttari-c',
    '7WJ94ZQ36Y': 'jazzmiles',
    'J9QGZ654YR': 'relax-c',
    '885X7WKYRN': 'chillout-c',
    'J1KJ59Q96X': 'piano-logue',
    'KZ5Q9LVMWL': 'rewind-time',
    'MJ885Z5MXR': 'my-favorite',
    'K1VRXRKXG6': 'hit-parade',
    'MK7LP4934M': 'izumi',
    '8ZRJYMZX4Z': 'radirutime',
    'NWYPY4N3WW': 'kogaku',
    'J8792PY43V': 'my-asa',
    '2QVV8Q6LV2': 'yuran',
    'ZG79L367QZ': 'hikigatari',
    'Z9WGYY3GP5': 'ml',
    'WW2Z47QY27': 'hitotoki',
    'P1LK3137Z2': 'hirunoikoi',
    'N8M9ZPVK4L': 'kayou',
    'G6J5L92M18': 'kaiteki',
    '5P6KW7QL6X': 'oto',
    'Y17XK7QVY6': 'endless',
    'X7R2P2PW5P': 'nradi',
    'Z9L1V2M24L': 'bescla',
    'YR96XR51MZ': 'roudokusekai',
    '368315KKP8': 'radio-bizeigo',
    '77RQWQX1L6': 'gendaieigo'
}

In [7]:
client = ProgramGuideV3(api_key=os.environ.get('NHK_API_KEY',''))

In [8]:
jst_zi = ZoneInfo('Asia/Tokyo')

In [9]:
dt_tday = datetime.now(jst_zi) + timedelta(days=1)
dt_tday = dt_tday.replace(hour=4, minute=0, second= 0, microsecond=0, tzinfo=jst_zi)
dt_tday

datetime.datetime(2026, 3, 15, 4, 0, tzinfo=zoneinfo.ZoneInfo(key='Asia/Tokyo'))

In [10]:
dt_tday.strftime('%Y-%m-%d')

'2026-03-15'

In [11]:
dt_yday = dt_tday - timedelta(days=1)
dt_yday.strftime('%Y-%m-%d')

'2026-03-14'

In [12]:
dt_pivot = dt_tday + timedelta(days=1)
dt_pivot

datetime.datetime(2026, 3, 16, 4, 0, tzinfo=zoneinfo.ZoneInfo(key='Asia/Tokyo'))

In [13]:
dt_start_pivot = dt_tday - timedelta(days=1)
dt_start_pivot

datetime.datetime(2026, 3, 14, 4, 0, tzinfo=zoneinfo.ZoneInfo(key='Asia/Tokyo'))

In [14]:
dt_yday, dt_pivot, dt_tday

(datetime.datetime(2026, 3, 14, 4, 0, tzinfo=zoneinfo.ZoneInfo(key='Asia/Tokyo')),
 datetime.datetime(2026, 3, 16, 4, 0, tzinfo=zoneinfo.ZoneInfo(key='Asia/Tokyo')),
 datetime.datetime(2026, 3, 15, 4, 0, tzinfo=zoneinfo.ZoneInfo(key='Asia/Tokyo')))

In [15]:
pytz_jst = pytz.timezone('Asia/Tokyo')
pytz_jst

<DstTzInfo 'Asia/Tokyo' LMT+9:19:00 STD>

In [16]:
pytz_jst.localize(dt_tday.replace(tzinfo=None))

datetime.datetime(2026, 3, 15, 4, 0, tzinfo=<DstTzInfo 'Asia/Tokyo' JST+9:00:00 STD>)

In [17]:
pytz_jst.localize(dt_yday.replace(tzinfo=None))

datetime.datetime(2026, 3, 14, 4, 0, tzinfo=<DstTzInfo 'Asia/Tokyo' JST+9:00:00 STD>)

In [18]:
pytz_jst.localize(dt_yday.replace(tzinfo=None)) < pytz_jst.localize(dt_tday.replace(tzinfo=None))

True

In [19]:
def to_jst(dt:datetime)->datetime:
    ret = pytz_jst.normalize(
        pytz_jst.localize(
            dt))
    return ret

In [20]:
round_time(dt_tday)

datetime.datetime(2026, 3, 15, 4, 0, tzinfo=zoneinfo.ZoneInfo(key='Asia/Tokyo'))

In [21]:
to_jst(dt_tday.replace(tzinfo=None))

datetime.datetime(2026, 3, 15, 4, 0, tzinfo=<DstTzInfo 'Asia/Tokyo' JST+9:00:00 STD>)

In [22]:
to_jst(dt_yday.replace(tzinfo=None))

datetime.datetime(2026, 3, 14, 4, 0, tzinfo=<DstTzInfo 'Asia/Tokyo' JST+9:00:00 STD>)

In [23]:
pg_infos_tday = client.pg_date_radio(area='120', service='r3', date=dt_tday).get('r3').get('publication')
#pg_infos_tday

In [24]:
pg_infos_yday = client.pg_date_radio(area='120', service='r3', date=dt_yday).get('r3').get('publication')
pg_infos_yday

[{'type': 'BroadcastEvent',
  'id': 'r3-120-2026031471065',
  'name': '音楽の泉\u3000ドボルザークの組曲「アメリカ」',
  'description': '\u3000',
  'startDate': '2026-03-14T05:00:03+09:00',
  'endDate': '2026-03-14T05:50:00+09:00',
  'location': {'id': '001', 'name': '東京'},
  'identifierGroup': {'broadcastEventId': 'r3-120-2026031471065',
   'radioEpisodeId': 'BXK5LYW5GZ',
   'radioEpisodeName': 'ドボルザークの組曲「アメリカ」',
   'radioSeriesId': 'MK7LP4934M',
   'radioSeriesName': '音楽の泉',
   'serviceId': 'r3',
   'areaId': '120',
   'stationId': '108',
   'date': '2026-03-14',
   'eventId': '71065',
   'genre': [{'id': '0402', 'name1': '音楽', 'name2': 'クラシック・オペラ'},
    {'id': '1002', 'name1': '趣味/教育', 'name2': '音楽・美術・工芸'}],
   'siteId': '0685'},
  'misc': {'displayVideoMode': 'none',
   'displayVideoRange': 'sdr',
   'displayAudioMode': [],
   'audioMode': [],
   'supportCaption': False,
   'supportSign': False,
   'supportHybridcast': False,
   'supportDataBroadcast': False,
   'isInteractive': False,
   'isChangeabl

In [25]:
publications = [item for sublist in [pg_infos_yday, pg_infos_tday] for item in sublist]
pubdic = extract_publications_by_date(publications, dt_tday, dt_pivot)
#len(pubdic), pubdic

In [26]:
dt1 = datetime.today().date()
dt2 = dt1 + timedelta(days=1)
dt3 = dt1 + timedelta(days=2)
dt4 = dt1 + timedelta(days=3)
dt5 = dt1 + timedelta(days=4)
dt6 = dt1 + timedelta(days=5)
dt7 = dt1 + timedelta(days=6)
dt8 = dt1 + timedelta(days=7)

In [27]:
pub1 = client.pg_date_radio('120','r3',dt1).get('r3').get('publication')
pub2 = client.pg_date_radio('120','r3',dt2).get('r3').get('publication')
pub3 = client.pg_date_radio('120','r3',dt3).get('r3').get('publication')
pub4 = client.pg_date_radio('120','r3',dt4).get('r3').get('publication')
pub5 = client.pg_date_radio('120','r3',dt5).get('r3').get('publication')
pub6 = client.pg_date_radio('120','r3',dt6).get('r3').get('publication')
pub7 = client.pg_date_radio('120','r3',dt7).get('r3').get('publication')
pub8 = client.pg_date_radio('120','r3',dt8).get('r3').get('publication')

In [28]:
pub1w = [item for sublist in [pub1,pub2,pub3,pub4,pub5,pub6,pub7] for item in sublist]
#len(pub1w), pub1w

In [29]:
[pub.get('name') for pub in pub1w[1:3+1]]

['らじるの時間\u3000ＮＨＫラジオ\u3000音声波再編について',
 '気象情報（関東甲信越）',
 '現代の音楽\u3000現代音楽\u3000１００年のレガシー５７\u3000エリオット・カーター']

In [30]:
[pub.get('name') for pub in pub1w[4:5+1]]

['みんなのうた「どこ？どこ？」／「こだまでしょうか」', '気象情報・交通情報（関東甲信越）']

In [31]:
dateutil.parser.parse(pg_infos_tday[0].get('startDate'), ignoretz=True)

datetime.datetime(2026, 3, 15, 5, 0, 3)

In [32]:
round_time(dateutil.parser.parse(pg_infos_tday[0].get('startDate')))

datetime.datetime(2026, 3, 15, 5, 0, tzinfo=tzoffset(None, 32400))

In [33]:
to_jst(round_time(dateutil.parser.parse(pg_infos_tday[0].get('startDate'), ignoretz=True)))

datetime.datetime(2026, 3, 15, 5, 0, tzinfo=<DstTzInfo 'Asia/Tokyo' JST+9:00:00 STD>)

In [34]:
pg_infos_in_yday = [pub for pub in pg_infos_yday
    if to_jst(round_time(dateutil.parser.parse(pub.get('startDate'), ignoretz=True))) > dt_yday
]

In [35]:
len(pg_infos_in_yday)

42

In [36]:
#[pub for pub in pg_infos_tday
#    if round_time(dateutil.parser.parse(pub.get('startDate'))) > dt_tday
#]

In [37]:
dt_tday.isoformat()

'2026-03-15T04:00:00+09:00'

In [38]:
round_time(dateutil.parser.parse(pg_infos_yday[1].get('startDate'), ignoretz=True)).isoformat()

'2026-03-14T05:50:00'

In [39]:
for broadcast_ev in publications:
    if get_series_id(broadcast_event=broadcast_ev) is not None:
        broadcast_ev['rec_flag'] = True
    else:
        broadcast_ev['rec_flag'] = False

In [40]:
idx_broadcast_event = []

# search broadcast_event
for idx, broadcast_event in enumerate(pubdic):

    # alias id
    alias_id = get_alias_id(broadcast_event) or get_series_id(broadcast_event)

    # keytitle
    keytitle = get_alias_id(broadcast_event) or get_series_id(broadcast_event)
    broadcast_event['keytitle'] = keytitle

    # series_url
    series_url = get_series_url(broadcast_event) or get_about_url(broadcast_event)

    # start time
    s_time = get_start_date(broadcast_event)
    s_time_tmp = round_time(dt = dateutil.parser.parse(s_time))
    broadcast_event["start_time"] = s_time_tmp.isoformat()
    start_time = s_time_tmp.strftime("%H:%M")

    # end time
    e_time = get_end_date(broadcast_event)
    e_time_tmp = round_time(dt = dateutil.parser.parse(e_time))
    broadcast_event["end_time"] = e_time_tmp.isoformat()

    # check keytitle
    keyflag = False
    if broadcast_event['keytitle'] in keytitles:
        keyflag = True

    # skip reruns
    if broadcast_event['keytitle'] in key_reruns and start_time in reruns.get(broadcast_event['keytitle']):
        keyflag = False
        continue

    broadcast_event['rec_flag'] = keyflag

#    print(keytitle)
    # rewrite sign title to named title
    if keytitle in signs.keys():
        keytitle = signs.get(keytitle)
#        print(keytitle)
        broadcast_event["keytitle"] = keytitle

#    print(keytitle)
    if broadcast_event['rec_flag'] == True:
#        print(broadcast_event.get('keytitle'))
        idx_broadcast_event.append(idx)

idx_broadcast_event

[16, 26, 29, 32, 37, 38, 40, 42]

In [41]:
consecutive_runs = extract_consecutive_runs(idx_broadcast_event)
consecutive_runs

[[16], [26], [29], [32], [37, 38], [40], [42]]

In [42]:
import json

In [43]:
pg = list()
for cons in consecutive_runs:
    num = len(cons)
    p = {
        "num":num,
        "list":[]
    }
    for i, idx in enumerate(cons):
        keytitle = pubdic[idx].get('keytitle')
        s_time = get_start_date(pubdic[idx])
        s_time_tmp = round_time(dt = dateutil.parser.parse(s_time))
        start_time = s_time_tmp.strftime("%H:%M")

        dstart = dateutil.parser.parse(start_time)
        dstart -= timedelta(minutes=1)

        e_time = get_end_date(pubdic[idx])
        e_time_tmp = round_time(dt = dateutil.parser.parse(e_time))
        end_time = e_time_tmp.strftime("%H:%M")

        dend = dateutil.parser.parse(end_time)

        if dend < dstart:
            dend += timedelta(hours=24)

        dur = dend - dstart - timedelta(seconds=30)
        durh = dur//timedelta(hours=1)
        durm = (dur/timedelta(hours=1) - durh) * timedelta(hours=1)
        durm = durm//timedelta(minutes=1)

        durd = 0
        if num > 1:
            if i == 0:
                durd = 45

        durs = 0
        if num > 1:
            if i == num - 1:
                durs = 30

        if num == 1:
            # overwrite
            durs = 30
            durd = 45

        dur = (keytitle, (dstart.month, dstart.day), (dstart.hour, dstart.minute), (dend.hour, dend.minute), durd, durh, durm, durs)
        dur = dict()
        dur['keytitle'] = keytitle
        dur['startDate'] = dstart.strftime("%M %H %d %m * 45")
        dur['endDate'] = (dend + timedelta(minutes=1)).strftime("%M %H %d %m")
        d_e = dend + timedelta(seconds=30) - dstart
        d_e1 = d_e // timedelta(seconds=60)
        d_e2 = (d_e - d_e1 * timedelta(seconds=60))//timedelta(seconds=1)
        dur['duration'] = "{:02d}:{:02d}".format(d_e1, d_e2)

        p['list'].append(dur)

    pg.append(p)
pg

[{'num': 1,
  'list': [{'keytitle': 'meiensou',
    'startDate': '59 08 14 03 * 45',
    'endDate': '56 10 14 03',
    'duration': '116:30'}]},
 {'num': 1,
  'list': [{'keytitle': 'kakecla',
    'startDate': '59 13 14 03 * 45',
    'endDate': '51 15 14 03',
    'duration': '111:30'}]},
 {'num': 1,
  'list': [{'keytitle': 'hibiki',
    'startDate': '59 15 14 03 * 45',
    'endDate': '51 16 14 03',
    'duration': '51:30'}]},
 {'num': 1,
  'list': [{'keytitle': 'hoshizora',
    'startDate': '59 16 14 03 * 45',
    'endDate': '01 18 14 03',
    'duration': '61:30'}]},
 {'num': 2,
  'list': [{'keytitle': 'bravo',
    'startDate': '24 19 14 03 * 45',
    'endDate': '26 20 14 03',
    'duration': '61:30'},
   {'keytitle': 'r-passio',
    'startDate': '24 20 14 03 * 45',
    'endDate': '01 21 14 03',
    'duration': '36:30'}]},
 {'num': 1,
  'list': [{'keytitle': 'jazzvoyage',
    'startDate': '49 21 14 03 * 45',
    'endDate': '41 22 14 03',
    'duration': '51:30'}]},
 {'num': 1,
  'list': 

In [44]:
[(idx, pubdic[idx].get('startDate'), get_alias_id(pubdic[idx]) or get_series_id(pubdic[idx])) for idx in idx_broadcast_event
	if dateutil.parser.parse(pubdic[idx].get('startDate')) >= dt_tday and
		dateutil.parser.parse(pubdic[idx].get('startDate')) < dt_pivot]


[(16, '2026-03-15T09:00:03+09:00', 'meiensou'),
 (26, '2026-03-15T14:00:03+09:00', 'kakecla'),
 (29, '2026-03-15T16:00:03+09:00', 'hibiki'),
 (32, '2026-03-15T17:00:03+09:00', 'hoshizora'),
 (37, '2026-03-15T19:25:00+09:00', 'bravo'),
 (38, '2026-03-15T20:25:00+09:00', 'r-passio'),
 (40, '2026-03-15T21:50:00+09:00', 'jazzvoyage'),
 (42, '2026-03-15T23:30:00+09:00', 'anata')]